In [13]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText
import os
from datetime import datetime
from dotenv import load_dotenv
from huggingface_hub import login
import pandas as pd

load_dotenv()
hf_token = os.getenv("HF_TOKEN")
login(token=hf_token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [9]:
def parse_in_json(llm_response):
    temp = eval(llm_response)
    if not isinstance(temp, dict):
        return {
            "time": 0.0,
            "coordinate": [[0, 0], [0, 0]],
            "type": "head-on",
            "why": "invalid response format"
        }
    return temp

def save_submission(submission, experiment_name):
    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    save_dir = os.path.join("result", f"{experiment_name}_{timestamp}")
    os.makedirs(save_dir, exist_ok=True)
    submission_path = os.path.join(save_dir, "submission.csv")
    description_path = os.path.join(save_dir, "description.txt")
    submission.to_csv(submission_path, index=False, lineterminator="\n")
    open(description_path, "w").close()
    print(f"saved to {save_dir}")

def make_submission(results):
    rows = []
    for result in results:
        video_path = result.get("video_path", "")
        path = "videos/" + video_path.split("/videos/")[-1] if "/videos/" in video_path else video_path
        accident_time = float(result.get("time", 0.0))
        coordinate = result.get("coordinate", [[0, 0], [0, 0]])
        (x1, y1), (x2, y2) = coordinate
        center_x = round(((x1 + x2) / 2) / 1000, 3)
        center_y = round(((y1 + y2) / 2) / 1000, 3)
        accident_type = result.get("type", "unknown")
        rows.append({
            "path": path,
            "accident_time": round(accident_time, 2),
            "center_x": center_x,
            "center_y": center_y,
            "type": accident_type
        })

    submission = pd.DataFrame(rows, columns=["path", "accident_time", "center_x", "center_y", "type"])
    return submission

In [3]:
class VideoInferenceVLM:
    def video_inference(self, video_path, prompt, max_new_tokens=128):
        raise NotImplementedError("Subclasses should implement this method.")

class Qwen3VLInference(VideoInferenceVLM):
    def __init__(self, model_id = "Qwen/Qwen3-VL-8b-Instruct"):

        self.model = AutoModelForImageTextToText.from_pretrained(
            model_id, dtype="auto", device_map="auto"
        )
        self.processor = AutoProcessor.from_pretrained(model_id)

    def video_inference(self, video_path, prompt, max_new_tokens=128):
        # Messages containing a video url(or a local path) and a text query
        messages = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "video",
                        "video": video_path,
                    },
                    {"type": "text", "text": prompt},
                ],
            }
        ]
        # Preparation for inference
        inputs = self.processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt"
        )
        inputs = inputs.to(self.model.device)

        # Inference: Generation of the output
        with torch.inference_mode():
            generated_ids = self.model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True)

        generated_ids_trimmed = [
            out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text = self.processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )
        return output_text

inference = Qwen3VLInference()

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

In [ ]:
instruction = "return the time when the collision occurs, " \
    "and return the collision bounding box with left-top and right-bottom coordinates. " \
    "And return the collision type. The collision type includes: head-on, rear-end, sideswipe, single, " \
    "and t-bone collisions. " \
    "Head-on is defined as a collision where the front ends of two vehicles hit each other. " \
    "Rear-end is defined as a collision where the front end of one vehicle hits the rear end of another vehicle. " \
    "Sideswipe is defined as a slight collision where the sides of two vehicles hit each other. " \
    "Single is defined as an accident that involves only one vehicle, such as a vehicle hitting a stationary object or a vehicle losing control and crashing without colliding with another vehicle. " \
    "T-bone is defined as a collision where the front end of one vehicle hits the side of another vehicle, forming a 'T' shape."

return_format = """
please return the result in JSON format only, not markdown.
here is the JSON format:
{
    "time": exact time, the temporal location of the video where collision occured,
    "coordinate": left-top and right-bottom, the position of bounding box on the video frame that contains the collision,
    "type": choose and return one of the following [head-on, rear-end, sideswipe, single, t-bone],
    "why": explain why did you return that time, coordinate and type.
}
---
example:
{
    "time": "mm.ss",
    "coordinate": [
        [x1, y1],
        [x2, y2]
    ],
    "type": "one of the following [head-on, rear-end, sideswipe, single, t-bone]"
    "why": " ... "
}
"""

# video_path = "/workspace/dataset/videos/__WFqm4i3vE_00.mp4"
# output = inference.video_inference(video_path, instruction + return_format, max_new_tokens=128)
# print(output[0])

In [14]:
test_path_file = "dataset/test_video_path.txt"
prompt = instruction + return_format
results = []

with open(test_path_file, "r") as f:
    video_paths = [line.strip() for line in f if line.strip()]

for i, video_path in enumerate(video_paths):
    output = inference.video_inference(
        video_path,
        prompt,
        max_new_tokens=128
    )
    output_json = parse_in_json(output[0])
    output_json["video_path"] = video_path
    results.append(output_json)
    print(f"{i+1}/{len(video_paths)} is done.")

/usr/local/lib/python3.10/dist-packages/transformers/video_processing_utils.py:872: UserWarning: `torchcodec` is not installed and cannot be used to decode the video by default. Falling back to `torchvision`. Note that `torchvision` decoding is deprecated and will be removed in future versions. 
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/video_utils.py:534: UserWarning: Using `torchvision` for video decoding is deprecated and will be removed in future versions. Please use `torchcodec` instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/io/_video_deprecation_warning.py:9: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, where we'll consolidate the future decoding/encoding capabilities of PyTorch: https://github.com/pytorch/torchcodec
  warnings.warn(


1/2027 is done.
2/2027 is done.
3/2027 is done.
4/2027 is done.
5/2027 is done.
6/2027 is done.
7/2027 is done.
8/2027 is done.
9/2027 is done.
10/2027 is done.
11/2027 is done.
12/2027 is done.
13/2027 is done.
14/2027 is done.
15/2027 is done.
16/2027 is done.
17/2027 is done.
18/2027 is done.
19/2027 is done.
20/2027 is done.
21/2027 is done.
22/2027 is done.
23/2027 is done.
24/2027 is done.
25/2027 is done.
26/2027 is done.
27/2027 is done.
28/2027 is done.
29/2027 is done.
30/2027 is done.
31/2027 is done.
32/2027 is done.
33/2027 is done.
34/2027 is done.
35/2027 is done.
36/2027 is done.
37/2027 is done.
38/2027 is done.
39/2027 is done.
40/2027 is done.
41/2027 is done.
42/2027 is done.
43/2027 is done.
44/2027 is done.
45/2027 is done.
46/2027 is done.
47/2027 is done.
48/2027 is done.
49/2027 is done.
50/2027 is done.
51/2027 is done.
52/2027 is done.
53/2027 is done.


KeyboardInterrupt: 

In [12]:
submission = make_submission(results)
save_submission(submission, "qwen3_vl_experiment")

saved to result/qwen3_vl_experiment_20260307064953
